# 2. Conventional Approaches to Phase Identification

In [ ]:
# @title Environment Setup
!pip install pymatgen numpy matplotlib scipy -q
print("Packages installed")

In [ ]:
# @title Load Tutorial Data
import os

REPO = "MRS_CH08_Tutorial"
REPO_URL = "https://github.com/Szymanski-Group/MRS_CH08_Tutorial.git"

if os.path.basename(os.getcwd()) == REPO:
    print("Data already present")
elif os.path.exists(REPO):
    os.chdir(REPO)
    print("Data already present")
else:
    !git clone {REPO_URL} -q
    os.chdir(REPO)
    print("Data loaded successfully")


## 2a) Search Match

Here we detect peaks in experimental patterns and rank candidate phases using FoM-style line matching scores.

In [ ]:
import csv
import glob
from pathlib import Path

from IPython.display import Image, display

from tutorial_utils.conventional import search_match as sm
from tutorial_utils.sections import conventional_search_match as vis_sm


def create_search_match_demo(
    experiment_dir="data/exp_patterns/one_phase",
    reference_dir="data/reference_structures",
    output_dir="outputs/conventional/search_match",
    top_k_to_print=3,
    max_experiment_patterns=4,
    min_angle=10.0,
    max_angle=100.0,
    plot_min_angle=10.0,
    plot_max_angle=80.0,
    wavelength="CuKa",
    wavelength_angstrom=1.5406,
    reference_intensity_threshold=1.0,
    baseline_percentile=5.0,
    peak_prominence_fraction=0.02,
    min_peak_distance_deg=0.22,
    max_detected_peaks=30,
    num_obs_lines_for_fom=20,
    match_tolerance_deg=0.25,
    min_matched_lines_for_score=6,
):
    """Step-by-step peak search-match demo using small specialized helper functions."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    vis_sm.OUTPUT_DIR = output_dir
    vis_sm.TOP_K_TO_PRINT = top_k_to_print
    vis_sm.PLOT_MIN_ANGLE = plot_min_angle
    vis_sm.PLOT_MAX_ANGLE = plot_max_angle

    exp_files = sorted(Path(experiment_dir).glob("*.xy"))
    if max_experiment_patterns is not None:
        exp_files = exp_files[:max_experiment_patterns]

    refs = sm.load_reference_library(
        sorted(Path(reference_dir).glob("*.cif")),
        min_angle=min_angle,
        max_angle=max_angle,
        wavelength=wavelength,
        intensity_threshold=reference_intensity_threshold,
    )

    all_rows = []
    print("\n=== Peak Search-Match Demo (de Wolff + Smith-Snyder) ===")
    print(f"Experimental patterns: {len(exp_files)}")
    print(f"Reference phases:      {len(refs)}")

    for exp_file in exp_files:
        pattern_name = exp_file.stem

        # Step 1: load and normalize profile
        tt, intensity = sm.load_pattern(exp_file, min_angle=min_angle, max_angle=max_angle)

        # Step 2: detect peaks
        _, obs_peaks = sm.detect_peaks(
            tt,
            intensity,
            baseline_percentile=baseline_percentile,
            prominence_fraction=peak_prominence_fraction,
            min_peak_distance_deg=min_peak_distance_deg,
            max_detected_peaks=max_detected_peaks,
        )

        # Step 3: compute FoM rankings
        by_dewolff, by_smith = sm.rank_phases(
            obs_peaks,
            refs,
            num_obs_lines_for_fom=num_obs_lines_for_fom,
            match_tolerance_deg=match_tolerance_deg,
            min_matched_lines_for_score=min_matched_lines_for_score,
            wavelength_angstrom=wavelength_angstrom,
        )

        print(f"\n--- {pattern_name} ---")
        print(f"Detected peaks: {len(obs_peaks)}")
        vis_sm.print_rank_table(pattern_name, by_dewolff, "de_wolff", "de Wolff")
        vis_sm.print_rank_table(pattern_name, by_smith, "smith_snyder", "Smith-Snyder")

        # Step 4: visualize best matches
        vis_sm.plot_summary(pattern_name, tt, intensity, obs_peaks, by_dewolff[0], by_smith[0], refs)

        for row in by_dewolff:
            all_rows.append({"pattern": pattern_name, "phase": row["phase"], **{k: row[k] for k in row if k != "phase"}})

    csv_file = output_dir / "all_pattern_rankings.csv"
    with open(csv_file, "w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "pattern",
                "phase",
                "de_wolff",
                "smith_snyder",
                "n_used",
                "n_match",
                "n_possible",
                "mean_delta_2theta",
            ],
        )
        writer.writeheader()
        writer.writerows(all_rows)

    print(f"\nSaved ranking table: {csv_file}")
    return all_rows


## Run the Demo
The next cell executes the shared tutorial module.

In [ ]:
# @title Run Search-Match Demo
create_search_match_demo()

# Try on your own:
# create_search_match_demo(match_tolerance_deg=0.20, min_peak_distance_deg=0.30)


## What To Observe
Compare top-ranked phases and where the reference sticks align with experimental peaks.

In [ ]:
for p in sorted(glob.glob("outputs/conventional/search_match/*_summary.png"))[:3]:
    display(Image(p))

## Summary
- Peak-list matching is fast and interpretable.
- Performance depends on peak detection quality and tolerance settings.
- This method can struggle when peaks overlap or broaden heavily.

## Next Steps
Continue to the next section below in this notebook.

## 02b - Profile Correlation

In [ ]:
import csv
import glob
from pathlib import Path

from IPython.display import Image, display

from tutorial_utils.conventional import profile_correlation as pc
from tutorial_utils.sections import conventional_profile_correlation as vis_pc


def create_profile_correlation_demo(
    experiment_dir="data/exp_patterns/one_phase",
    reference_dir="data/reference_structures",
    output_dir="outputs/conventional/profile_correlation",
    top_k_to_print=3,
    max_experiment_patterns=4,
    min_angle=10.0,
    max_angle=80.0,
    wavelength="CuKa",
    reference_intensity_threshold=1.0,
    fwhm=0.30,
    gauss_frac=0.2,
    baseline_percentile=5.0,
):
    """Step-by-step profile-correlation demo using small specialized helper functions."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    vis_pc.OUTPUT_DIR = output_dir
    vis_pc.TOP_K_TO_PRINT = top_k_to_print
    vis_pc.FWHM = fwhm
    vis_pc.GAUSS_FRAC = gauss_frac

    exp_files = sorted(Path(experiment_dir).glob("*.xy"))
    if max_experiment_patterns is not None:
        exp_files = exp_files[:max_experiment_patterns]

    ref_lib = pc.load_reference_stick_library(
        sorted(Path(reference_dir).glob("*.cif")),
        min_angle=min_angle,
        max_angle=max_angle,
        wavelength=wavelength,
        intensity_threshold=reference_intensity_threshold,
    )

    all_rows = []
    print("\n=== Full-Profile Correlation Demo (Pearson + Cosine) ===")
    print(f"Experimental patterns: {len(exp_files)}")
    print(f"Reference phases:      {len(ref_lib)}")

    for exp_file in exp_files:
        pattern_name = exp_file.stem

        # Step 1: preprocess experimental profile
        two_theta, exp_profile = pc.load_experimental_profile(
            exp_file,
            min_angle=min_angle,
            max_angle=max_angle,
            baseline_percentile=baseline_percentile,
        )

        # Step 2: simulate each candidate and compute similarity
        by_pearson, by_cosine, simulated_profiles = pc.rank_phases(
            exp_profile,
            two_theta,
            ref_lib,
            fwhm=fwhm,
            gauss_frac=gauss_frac,
        )

        print(f"\n--- {pattern_name} ---")
        vis_pc.print_rank_table(pattern_name, by_pearson, "pearson", "Pearson")
        vis_pc.print_rank_table(pattern_name, by_cosine, "cosine", "Cosine")

        # Step 3: visualize best matches
        vis_pc.plot_summary(pattern_name, two_theta, exp_profile, by_pearson[0], by_cosine[0], simulated_profiles)

        for row in by_pearson:
            all_rows.append({"pattern": pattern_name, "phase": row["phase"], "pearson": row["pearson"], "cosine": row["cosine"]})

    csv_file = output_dir / "all_pattern_profile-correlations.csv"
    with open(csv_file, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["pattern", "phase", "pearson", "cosine"])
        writer.writeheader()
        writer.writerows(all_rows)

    print(f"\nSaved ranking table: {csv_file}")
    return all_rows


# 02b — Full-Profile Correlation

Instead of line matching, this method compares full simulated and observed profiles via correlation metrics.

## Run the Demo
The next cell executes the shared tutorial module.

In [ ]:
# @title Run Profile-Correlation Demo
create_profile_correlation_demo()

# Try on your own:
# create_profile_correlation_demo(fwhm=0.45, gauss_frac=0.35)


## What To Observe
Look for differences between Pearson and cosine rankings on the same pattern.

In [ ]:
for p in sorted(glob.glob("outputs/conventional/profile_correlation/*_profile-correlation.png"))[:3]:
    display(Image(p))

## Summary
- Profile-level comparison uses more information than discrete peak lists.
- Pearson and cosine can prioritize slightly different candidates.
- Baseline handling strongly influences correlation scores.

## Next Steps
Continue to the next section below in this notebook.

## 02c - Rietveld Refinement

In [ ]:
import csv
import glob
from pathlib import Path

from IPython.display import Image, display

from tutorial_utils.conventional import rietveld as rv
from tutorial_utils.sections import conventional_rietveld as vis_rv


def create_rietveld_demo(
    experiment_dir="data/exp_patterns/one_phase",
    reference_dir="data/reference_structures",
    output_dir="outputs/conventional/rietveld",
    top_k_to_print=3,
    patterns_to_run=("TiO2", "ZrO2"),
    min_angle=10.0,
    max_angle=80.0,
    wavelength="CuKa",
    reference_intensity_threshold=1.0,
    baseline_percentile=5.0,
    background_degree=6,
    fwhm_init=0.30,
    gauss_frac=0.2,
    lattice_scale_bounds=(0.98, 1.02),
    fwhm_bounds=(0.05, 1.20),
    lattice_maxiter=60,
    width_maxiter=50,
):
    """Step-by-step sequential Rietveld-style refinement using specialized helper functions."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    vis_rv.OUTPUT_DIR = output_dir
    vis_rv.TOP_K_TO_PRINT = top_k_to_print

    exp_files = sorted(Path(experiment_dir).glob("*.xy"))
    if patterns_to_run is not None:
        keep = set(patterns_to_run)
        exp_files = [f for f in exp_files if f.stem in keep]

    structures = rv.load_reference_structures(sorted(Path(reference_dir).glob("*.cif")))
    calculator = rv.XRDCalculator(wavelength=wavelength)

    all_rows = []
    print("\n=== Sequential Rietveld-Style Demo ===")
    print(f"Experimental patterns: {len(exp_files)}")
    print(f"Reference phases:      {len(structures)}")

    for exp_file in exp_files:
        pattern_name = exp_file.stem

        # Step 1: preprocess experimental profile
        two_theta, y_obs = rv.load_experimental_profile(
            exp_file,
            min_angle=min_angle,
            max_angle=max_angle,
            baseline_percentile=baseline_percentile,
        )

        # Step 2: sequentially refine each candidate phase
        rows = []
        for phase, structure in structures.items():
            result = rv.refine_phase_sequential(
                two_theta,
                y_obs,
                structure,
                calculator,
                min_angle=min_angle,
                max_angle=max_angle,
                intensity_threshold=reference_intensity_threshold,
                background_degree=background_degree,
                fwhm_init=fwhm_init,
                gauss_frac=gauss_frac,
                lattice_scale_bounds=lattice_scale_bounds,
                fwhm_bounds=fwhm_bounds,
                lattice_maxiter=lattice_maxiter,
                width_maxiter=width_maxiter,
            )
            rows.append({"phase": phase, **result})

        # Step 3: rank by refined fit quality
        rows.sort(key=lambda r: r["rwp"])

        print(f"\n--- {pattern_name} ---")
        vis_rv.print_rank_table(pattern_name, rows)

        # Step 4: visualize best refinement sequence
        vis_rv.plot_refinement_summary(pattern_name, two_theta, y_obs, rows[0])

        for row in rows:
            s = row["scales"]
            all_rows.append(
                {
                    "pattern": pattern_name,
                    "phase": row["phase"],
                    "rwp": row["rwp"],
                    "pearson": row["pearson"],
                    "a_scale": s[0],
                    "b_scale": s[1],
                    "c_scale": s[2],
                    "fwhm": row["fwhm"],
                }
            )

    csv_file = output_dir / "all_pattern_rietveld-sequential.csv"
    with open(csv_file, "w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["pattern", "phase", "rwp", "pearson", "a_scale", "b_scale", "c_scale", "fwhm"],
        )
        writer.writeheader()
        writer.writerows(all_rows)

    print(f"\nSaved ranking table: {csv_file}")
    return all_rows


# 02c — Sequential Rietveld-Style Refinement

This simplified workflow refines background, lattice scales, and width parameters in sequence for each candidate phase.

## Run the Demo
The next cell executes the shared tutorial module.

In [ ]:
# @title Run Sequential Rietveld Demo
create_rietveld_demo()

# Try on your own:
# create_rietveld_demo(patterns_to_run=("LiMnO2",), background_degree=4, fwhm_init=0.22)


## What To Observe
Track how each refinement stage improves the fit and lowers Rwp.

In [ ]:
for p in sorted(glob.glob("outputs/conventional/rietveld/*_rietveld-sequential.png")):
    display(Image(p))

## Summary
- Sequential refinement isolates effects of key parameter groups.
- Rwp provides a compact fit-quality ranking.
- Refinement-based methods are accurate but more compute-intensive.

## Next Steps
Continue to **03 — ML + Deep Learning**: [Open in Colab](https://colab.research.google.com/github/Szymanski-Group/MRS_CH08_Tutorial/blob/main/notebooks/03_ML-Deep-Learning.ipynb)